# First rule of Silver: inspect before cleaning

we're reading an existing Databricks table

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_cust = spark.table(
    "e2e_project.bronze.crm_cust_info"
)

display(bronze_cust)

## Inspect its schema

In [0]:
bronze_cust.printSchema()

In [0]:
print("Total rows:", bronze_cust.count())

In [0]:
print(
    "Distinct customer IDs:",
    bronze_cust.select("cst_id").distinct().count()
)

In [0]:
duplicates = (
    bronze_cust
    .groupBy("cst_id")
    .count()
    .filter(F.col("count") > 1)
)

display(duplicates)

We use the duplicate-ID list to retrieve the complete customer records.

In [0]:
duplicate_rows = (
    bronze_cust
    .join(
        duplicates.select("cst_id"),
        on="cst_id",
        how="inner"
    )
    .orderBy("cst_id", "cst_create_date")
)

display(duplicate_rows)


cst_id          → 0 null values

cst_firstname   → 4 null values

cst_gndr        → 7 null values


This is data profiling

In [0]:
null_counts = bronze_cust.select(
    [
        F.sum(
            F.col(c).isNull().cast("int")
        ).alias(c)
        for c in bronze_cust.columns
    ]
)

display(null_counts)

## Check whitespace problems

In [0]:
display(
    bronze_cust
    .filter(
        (F.col("cst_firstname") != F.trim(F.col("cst_firstname"))) |
        (F.col("cst_lastname") != F.trim(F.col("cst_lastname")))
    )
)

## Inspect categorical values

In [0]:
display(
    bronze_cust
    .groupBy("cst_marital_status")
    .count()
)

In [0]:
display(
    bronze_cust
    .groupBy("cst_gndr")
    .count()
)

## creating the clean customer DataFrame

Only after profiling do we transform.

First, deduplicate customers.

We want to keep the most recent record for each cst_id.

for every customer ID

        ↓

sort its records
by create_date descending

        ↓

latest record comes first

In [0]:
customer_window = (
    Window
    .partitionBy("cst_id")
    .orderBy(
        F.col("cst_create_date").desc_nulls_last()
    )
)

In [0]:
silver_cust = (
    bronze_cust

    # Customer ID must exist
    .filter(F.col("cst_id").isNotNull())

    # Rank duplicate records
    .withColumn(
        "_row_number",
        F.row_number().over(customer_window)
    )

    # Keep latest customer record
    .filter(F.col("_row_number") == 1)

    # Remove temporary ranking column
    .drop("_row_number")
)

This is much better than:

dropDuplicates(["cst_id"])

because dropDuplicates() doesn't express which record we intentionally want.

Here we explicitly say:

Keep the latest customer record.

## Clean names

In [0]:
silver_cust = (
    silver_cust
    .withColumn(
        "cst_firstname",
        F.trim(F.col("cst_firstname"))
    )
    .withColumn(
        "cst_lastname",
        F.trim(F.col("cst_lastname"))
    )
)

## Standardize marital status

In [0]:
silver_cust = silver_cust.withColumn(
    "cst_marital_status",
    F.when(
        F.upper(F.trim(F.col("cst_marital_status"))) == "S",
        "Single"
    )
    .when(
        F.upper(F.trim(F.col("cst_marital_status"))) == "M",
        "Married"
    )
    .otherwise("n/a")
)

## Standardize gender

In [0]:
silver_cust = silver_cust.withColumn(
    "cst_gndr",
    F.when(
        F.upper(F.trim(F.col("cst_gndr"))) == "F",
        "Female"
    )
    .when(
        F.upper(F.trim(F.col("cst_gndr"))) == "M",
        "Male"
    )
    .otherwise("n/a")
)

##Make sure the date is a real date

In [0]:
silver_cust = silver_cust.withColumn(
    "cst_create_date",
    F.to_date(F.col("cst_create_date"))
)

## Inspect the result before saving

In [0]:
display(silver_cust)

In [0]:
silver_cust.printSchema()

## Validate uniqueness

In [0]:
remaining_duplicates = (
    silver_cust
    .groupBy("cst_id")
    .count()
    .filter(F.col("count") > 1)
)

display(remaining_duplicates)

In [0]:
print("Silver rows:", silver_cust.count())

print(
    "Distinct IDs:",
    silver_cust.select("cst_id").distinct().count()
)

## Validate standardized categories

In [0]:
display(
    silver_cust
    .groupBy("cst_marital_status")
    .count()
)

In [0]:
display(
    silver_cust
    .groupBy("cst_gndr")
    .count()
)

## Write the first Silver Delta table

In [0]:
(
    silver_cust.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "e2e_project.silver.crm_cust_info"
    )
)

BRONZE.crm_cust_info


Duplicate customers

Encoded categories

Whitespace

Possible nulls

Raw source structure


             ↓

     PySpark cleaning


             ↓

SILVER.crm_cust_info


1 row/customer

Clean names

Single / Married

Female / Male

Correct date type

Validated data